# 📘 이미지 처리 기법

이미지 처리의 핵심 기법을 배웁니다: 리사이징, 크롭, 필터링, 임계값 처리, 엣지 검출 등.

**학습 목표:**
- 이미지 리사이징과 크롭
- 블러, 샤프닝 등 필터링
- 임계값 처리(이진화)
- 엣지 검출(Canny)
- 모폴로지 연산

## 1. 리사이징과 크롭

이미지의 크기를 변환하거나 원하는 영역을 잘라내는 기본 작업입니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  이미지 리사이징과 크롭                  │
# └─────────────────────────────────────────┘

# 테스트 이미지 생성 (체커보드 + 원)
img = np.zeros((400, 600, 3), dtype=np.uint8)
img[:] = (240, 240, 240)  # 밝은 회색 배경 (BGR)

# 체커보드 패턴
for i in range(0, 600, 50):
    for j in range(0, 400, 50):
        if (i // 50 + j // 50) % 2 == 0:
            img[j:j+50, i:i+50] = (200, 200, 200)

# 원 그리기
cv2.circle(img, (300, 200), 80, (0, 100, 255), -1)  # 주황 원
cv2.rectangle(img, (100, 100), (200, 300), (255, 0, 0), 3)  # 파란 사각형

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 리사이징
img_half = cv2.resize(img_rgb, (300, 200))  # 절반 크기
img_double = cv2.resize(img_rgb, (1200, 800))  # 2배 크기
img_fx = cv2.resize(img_rgb, None, fx=0.5, fy=0.5)  # 비율로 지정

# 크롭 (ROI 추출)
cropped = img_rgb[100:300, 100:400]  # y: 100~300, x: 100~400

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0, 0].imshow(img_rgb); axes[0, 0].set_title(f'원본 {img_rgb.shape}')
axes[0, 1].imshow(img_half); axes[0, 1].set_title(f'절반 크기 {img_half.shape}')
axes[1, 0].imshow(img_double); axes[1, 0].set_title(f'2배 크기 {img_double.shape}')
axes[1, 1].imshow(cropped); axes[1, 1].set_title(f'크롭 {cropped.shape}')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. 필터링 — 블러와 샤프닝

필터링은 이미지를 부드럽게 하거나 선명하게 만드는 작업입니다.
커널(필터 마스크)을 이미지 위를 슬라이드하며 계산합니다.

| 필터 | 효과 | 함수 |
|------|------|------|
| 가우시안 블러 | 노이즈 제거, 부드러움 | `cv2.GaussianBlur()` |
| 박스 블러 | 단순 평균 블러 | `cv2.blur()` |
| 샤프닝 | 에지 강조 | 커널 직접 정의 |
| 미디안 블러 | 소금/후추 노이즈 제거 | `cv2.medianBlur()` |

In [ ]:
# ┌─────────────────────────────────────────┐
# │  필터링 — 블러와 샤프닝                 │
# └─────────────────────────────────────────┘

# 노이즈가 있는 이미지 생성
rng = np.random.default_rng(42)
img_clean = np.ones((200, 300, 3), dtype=np.uint8) * 180
cv2.circle(img_clean, (150, 100), 50, (0, 200, 255), -1)

# 소금/후추 노이즈 추가
img_noisy = img_clean.copy()
noise = rng.integers(0, 100, (200, 300))
img_noisy[noise < 3] = 0    # 검은 점
img_noisy[noise > 97] = 255  # 흰 점

# 필터링 적용
img_gaussian = cv2.GaussianBlur(img_noisy, (15, 15), 0)
img_median = cv2.medianBlur(img_noisy, 5)
img_box = cv2.blur(img_noisy, (15, 15))

# 샤프닝 커널
kernel_sharpen = np.array([[0, -1, 0],
                            [-1, 5, -1],
                            [0, -1, 0]])
img_sharp = cv2.filter2D(img_gaussian, -1, kernel_sharpen)

imgs = [img_noisy, img_gaussian, img_median, img_sharp]
titles = ['노이즈 원본', '가우시안 블러', '미디안 블러', '샤프닝']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, im, t in zip(axes.flat, imgs, titles):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(t)
    ax.axis('off')
plt.tight_layout()
plt.show()

print("💡 가우시안 블러: 일반적인 노이즈 제거에 적합")
print("💡 미디안 블러: 소금/후추 노이즈에 특히 효과적")
print("💡 샤프닝: 블러 처리 후 에지를 다시 선명하게")

## 3. 임계값 처리 (이진화)

임계값 처리는 이미지를 흑과 백 두 가지로 나누는 작업입니다.
객체와 배경을 분리할 때 사용합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  임계값 처리 (이진화)                    │
# │  그레이스케일 이미지를 흑백으로 변환      │
# └─────────────────────────────────────────┘

# 그라데이션 이미지 생성
x = np.linspace(0, 255, 300).astype(np.uint8)
img_grad = np.tile(x, (200, 1))

# 다양한 임계값 처리
ret1, thresh_binary = cv2.threshold(img_grad, 127, 255, cv2.THRESH_BINARY)
ret2, thresh_binary_inv = cv2.threshold(img_grad, 127, 255, cv2.THRESH_BINARY_INV)
ret3, thresh_trunc = cv2.threshold(img_grad, 127, 255, cv2.THRESH_TRUNC)
ret4, thresh_tozero = cv2.threshold(img_grad, 127, 255, cv2.THRESH_TOZERO)

# 적응형 임계값 (노이즈 이미지)
img_text = np.ones((200, 300), dtype=np.uint8) * 200
cv2.putText(img_text, 'OpenCV', (30, 120), cv2.FONT_HERSHEY_SIMPLEX, 3, 50, 5)

# Otsu 이진화
ret_otsu, thresh_otsu = cv2.threshold(img_text, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes[0, 0].imshow(img_grad, cmap='gray'); axes[0, 0].set_title('원본 그라데이션')
axes[0, 1].imshow(thresh_binary, cmap='gray'); axes[0, 1].set_title('BINARY (127)')
axes[0, 2].imshow(thresh_binary_inv, cmap='gray'); axes[0, 2].set_title('BINARY_INV')
axes[1, 0].imshow(thresh_trunc, cmap='gray'); axes[1, 0].set_title('TRUNC')
axes[1, 1].imshow(thresh_tozero, cmap='gray'); axes[1, 1].set_title('TOZERO')
axes[1, 2].imshow(thresh_otsu, cmap='gray'); axes[1, 2].set_title(f'Otsu (threshold={ret_otsu:.0f})')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Otsu 자동 임계값: {ret_otsu:.1f}")
print("\n💡 BINARY: 임계값 이상은 흰색, 미만은 검은색")
print("💡 Otsu: 히스토그램을 분석해 최적 임계값을 자동 계산")

## 4. 엣지 검출과 모폴로지

**엣지 검출**은 이미지에서 경계선을 찾는 기술입니다.
**모폴로지 연산**은 이진화된 이미지의 형태를 다듬습니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  엣지 검출과 모폴로지 연산               │
# └─────────────────────────────────────────┘

# 테스트 이미지 (도형)
img_shapes = np.zeros((300, 400), dtype=np.uint8)
cv2.rectangle(img_shapes, (30, 30), (170, 170), 255, -1)
cv2.circle(img_shapes, (300, 150), 80, 255, -1)
cv2.line(img_shapes, (200, 30), (380, 270), 255, 5)

# Canny 엣지 검출
edges1 = cv2.Canny(img_shapes, 50, 150)
edges2 = cv2.Canny(img_shapes, 100, 200)

# 모폴로지 연산
kernel = np.ones((5, 5), np.uint8)
dilated = cv2.dilate(img_shapes, kernel, iterations=3)  # 팽창
eroded = cv2.erode(img_shapes, kernel, iterations=3)     # 침식
opened = cv2.morphologyEx(img_shapes, cv2.MORPH_OPEN, kernel)  # 열기
closed = cv2.morphologyEx(img_shapes, cv2.MORPH_CLOSE, kernel) # 닫기

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
titles = ['원본', 'Canny(50,150)', 'Canny(100,200)', '팽창(dilate)',
          '', '침식(erode)', '열기(open)', '닫기(close)']
images = [img_shapes, edges1, edges2, dilated,
          None, eroded, opened, closed]
for ax, im, t in zip(axes.flat, images, titles):
    if im is not None:
        ax.imshow(im, cmap='gray')
        ax.set_title(t)
    ax.axis('off')
axes[1, 0].axis('off')
plt.tight_layout()
plt.show()

print("💡 Canny: 가장 널리 쓰이는 엣지 검출 알고리즘")
print("💡 팽창(dilate): 객체를 확대, 작은 구멍을 채움")
print("💡 침식(erode): 객체를 축소, 작은 노이즈를 제거")
print("💡 열기(open): 침식→팽창, 노이즈 제거에 효과적")
print("💡 닫기(close): 팽창→침식, 구멍 채우기에 효과적")

## 🎯 연습 문제

1. 이미지에 가우시안 노이즈를 추가하고, 세 가지 블러 필터로 노이즈를 제거해 비교하세요.
2. Otsu 임계값과 적응형 임계값(`cv2.adaptiveThreshold`)의 결과를 비교하세요.
3. Canny 엣지 검출의 임계값(threshold1, threshold2)을 변경하며 결과를 비교하세요.
4. 모폴로지 열기와 닫기를 순서대로 적용하여 효과를 확인하세요.
5. 이미지를 회전(`cv2.rotate`)하고 뒤집기(`cv2.flip`)하는 코드를 작성하세요.